Evaluation on how to use taf library taf.collector.l1l2bip.

The documentation of the collector itself is here: https://docs.taf.nsn-rdnet.net/taf.collectors.l1l2bip/

And the Commands for the bash are described here: https://confluence.ext.net.nokia.com/pages/viewpage.action?pageId=1014283497

In [ ]:
*** Settings ***
Library   OperatingSystem
Library   String

*** Keywords ***
Get Hostname for Settings
    ${ute_config}=   Get Environment Variable     STY
    ${_}   ${_}    ${hostname}=    Split String   ${ute_config}   separator=.
    ${hostname}=   Replace String    ${hostname}   -   _
    [Return]   ${hostname}

In [ ]:
*** Settings ***
Variables    /home/ute/testline_configuration/2_16_UTE5GCLASSICAL1815

*** Keywords ***
Get all the TL
    Log To Console    ${tl}
    [Return]    HostPC is ${tl.test_pc}

Use the bts_info annd fbb_info from sbts admin library to get ther horizontal and vertical position of the baseband module:


In [ ]:
*** Settings ***
Resource    ../gnb.robot
Resource    ../connect.robot
Library    taf.sbts.oam.coam.admin          WITH NAME   admin
Library    String

*** Keywords ***
Get Slot Of Baseband Module

    ${oam_alias}=    connect.Setup OAM http session
    ${bts_info}=    admin.Get Bts Info    connection=${oam_alias}
    ${baseband_module_distname}=    Set Variable    ${bts_info['basebandModules'][0]['id']['internal']}
    
    ${fbb_info}=    admin.get parameters   dist_name=${baseband_module_distname}    connection=${oam_alias}
    ${horizontalPosition}=  Set Variable  ${fbb_info['horizontalPosition']}
    ${verticalPosition}=    Set Variable  ${fbb_info['verticalPosition']}
    ${productName}=     Split String   ${fbb_info['productName']}
    ${product}=     Set Variable   ${productName[0]}
    Log    Product is ${product}   console=True

    ${slot}=               Evaluate   (${horizontalPosition}-1)*4 + (${verticalPosition} - 1)
    ${slot_board_mapping}=   Create Dictionary    ${slot}    ${product}
    
    [Return]  ${slot}   ${verticalPosition}
    
    [Teardown]    Teardown OAM http session




Functionality to create the calls on the lfs script for port mirroring:


In [ ]:
*** Settings ***
Library   Collections

*** Keywords ***
Get Commands to Mirror Backplane Port
    [Arguments]     ${backplane_port}   ${slot}
    ${commands}=   Create List
    ...   /usr/bin/lfs-nw_switch-mirror-tool.sh -p 0 -d ethsei02 -f L2HI_L2RT -b "${backplane_port}" -s ${slot} -l eths3 -v 303
    ...   /usr/bin/lfs-nw_switch-mirror-tool.sh -p 0 -d ethsei02 -f L1_L2_UL -b "${backplane_port}" -s ${slot} -l eths3 -v 304
    ...   /usr/bin/lfs-nw_switch-mirror-tool.sh -p 0 -d ethsei02 -f L1_L2_DL -b "${backplane_port}" -s ${slot} -l eths3 -v 305
    ...   ncp_portmirror -p xgmac96 -r
    ...   ncp_portmirror -d netdriver -p xgmac96
    [Return]   ${commands}


Get ABIO to ASIB Mirror Commands for Slot
    ${slot}   ${vertical}=   Get Slot Of Baseband Module
    
    &{port_backplane_map}=   Create Dictionary    1   C${vertical}B1(2)
    ...                                           2   C${vertical}B2(2)                        
    ...                                           3   C${vertical}B3(2)                        
    ...                                           5   C${vertical}B4(2)                        
    ...                                           6   C${vertical}B5(2)                        
    ...                                           7   C${vertical}B6(2)                        
    
    Log Dictionary   ${port_backplane_map}
    
    ${commands}=    Get Commands to Mirror Backplane Port   ${port_backplane_map['${slot}']}
    [Return]    ${commands}



Example of collection the ABIO traffic with the following KEyword:

In [ ]:
*** Settings ***
Library    taf.collectors.l1l2bip    WITH NAME    l1l2bip
Library    taf.collectors.tcpdump    WITH NAME    tcpdump
Library    taf.collectors.core       WITH NAME    core

*** Test Cases ***
Get L1_L2_UL and L1_L2_DL ABIO traffic to ASIB cpu
    ${asib}=  Create Dictionary
    ...    host=192.168.255.1
    ...    user=toor4nsn
    ...    password=oZPS0POrRieRtu
    tcpdump.create_tcpdump_collector
    ...    name=tcpdump
    ...    options=-v -i eths0 -C 100 -W 20
    ...    executor=${asib}
    
    ${asib_commands}=   Get ABIO to ASIB Mirror Commands for Slot
    
    l1l2bip.create_generic_l1l2bip_collector
    ...    name=Genericl1l2bip
    ...    collector=tcpdump
    ...    executor=${asib}
    ...    setup_commands=${asib_commands}
    core.setup_collectors    names=Genericl1l2bip
    core.start_collectors    names=Genericl1l2bip
        
    sleep    10s
    
    core.stop_collectors    names=Genericl1l2bip
    core.collect_collectors    names=Genericl1l2bip
    core.teardown_collectors    names=Genericl1l2bip